# Final Test Evaluation

The single, final evaluation on the **December test set** — held out and untouched
through all feature engineering, tuning, and model selection. This is the honest
estimate of real-world performance.

We load the saved model and the train-fit encoders, transform the test set with
those encoders (never refit on test), and report threshold-independent ranking
metrics plus leakage and no-skill sanity checks.

## Load test and validation sets

Load the December test split (and November validation for reference). The test
set has not been seen by any earlier step.

In [1]:
import pandas as pd
import joblib
from pyprojroot import here
from features import transform_features

test_df = pd.read_parquet(
    here("data/processed/test.parquet")
)
val_df = pd.read_parquet(
    here("data/processed/val.parquet")
)

## Load saved artifacts

Load the final model and the encoders fit during training. These come straight
from disk — we do not retrain anything here.

In [2]:
model = joblib.load(
    here("models/xgb_final.pkl")
)

encoders = joblib.load(
    here("models/encoders.pkl")
)

## Transform with train-fit encoders

`transform_features` applies the **train-fit** delay-rate mappings to the test
set. The encoders are never refit on test data — doing so would leak test
information into the features. This is the same apply path used at training and
serving, so features are identical across all three.

In [3]:
X_test = transform_features(
    test_df,
    encoders
)

y_test = test_df["DepDel15"]

X_val = transform_features(
    val_df,
    encoders
)

y_val = val_df["DepDel15"]

## Generate predictions

Produce predicted probabilities, then a binary label. Note: the 0.5 cutoff here
is a **placeholder**, not the operating threshold. Because the probabilities are
centered near the ~17% base rate, 0.5 is far too high — its purpose below is to
demonstrate exactly that. The real threshold must be chosen on validation
(see Next Steps).

In [4]:
test_probs = model.predict_proba(X_test)[:, 1]

test_preds = (test_probs >= 0.5).astype(int)

## Ranking metrics (threshold-independent)

- **Test PR-AUC: 0.327** against a no-skill floor of **0.216** (December's delay
  rate) — a lift of ~0.11 over baseline.
- **Test ROC-AUC: 0.654** — base-rate-independent, so the cleanest read of pure
  ranking ability: real but modest discriminative power.

Important: the test PR-AUC (0.327) is **not** directly comparable to the
validation PR-AUC (0.225). PR-AUC scales with the positive rate, and December
(21.6% delayed) is more imbalanced toward delays than November (14.4%). Measured
as *lift over each set's own floor*, validation (+0.08) and test (+0.11) are
consistent — the model generalizes without collapse, which is the result that
matters.

In [5]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

pr_auc = average_precision_score(
    y_test,
    test_probs
)

roc_auc = roc_auc_score(
    y_test,
    test_probs
)

print(f"Test PR-AUC : {pr_auc:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")

Test PR-AUC : 0.3270
Test ROC-AUC: 0.6538


## Classification report at the 0.5 placeholder threshold

At a 0.5 cutoff, recall on delayed flights is ~1% — the model almost never
predicts "delayed." This is **expected, not a defect**: predicted probabilities
sit near the ~17% base rate, so virtually none exceed 0.5. It is the clearest
possible evidence that 0.5 is the wrong operating point. The valid measures of
model quality are the threshold-independent PR-AUC / ROC-AUC above; the usable
precision/recall trade-off is set by the threshold chosen on validation.

In [6]:
print(
    classification_report(
        y_test,
        test_preds
    )
)

              precision    recall  f1-score   support

         0.0       0.78      1.00      0.88    458558
         1.0       0.47      0.01      0.01    126274

    accuracy                           0.78    584832
   macro avg       0.63      0.50      0.45    584832
weighted avg       0.72      0.78      0.69    584832



## Confusion matrix (0.5 threshold)

Confirms the same point numerically: of ~126k actual delays, only ~840 are caught
at 0.5. A correctly chosen threshold will move this trade-off toward useful
recall at the cost of some precision.

In [7]:
cm = confusion_matrix(
    y_test,
    test_preds
)

print(cm)

[[457624    934]
 [125435    839]]


## Save test predictions

Persist per-flight actuals, probabilities, and labels for error analysis and
later slice-based evaluation.

In [8]:
results = pd.DataFrame({
    "actual": y_test,
    "pred_prob": test_probs,
    "pred_label": test_preds
})

results.to_csv(
    here("models/test_predictions.csv"),
    index=False
)